<a href="https://colab.research.google.com/github/FeLiXp90/team-combinatorial-optimization-ai/blob/main/01_data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalação das Dependências

Instala o Google Chrome no ambiente Linux do Colab e as bibliotecas Python necessárias.

In [1]:
# 1. Adiciona o repositório oficial do Google Chrome e instala
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update -y
!apt-get install -y google-chrome-stable

# 2. Instala as bibliotecas de Web Scraping para Python
!pip install selenium beautifulsoup4 webdriver-manager -q

OK
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/

# A Classe: NarutoArenaScraper



Define a estrutura limpa e orientada a objetos do nosso scraper.

In [2]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By


class NarutoArenaScraper:
  """Classe responsável pela navegação automatizada e extração do HTML

  no Naruto-Arena utilizando Selenium Headless.
  """

  def __init__(self, base_url: str = 'https://www.naruto-arena.site/'):
    self.base_url = base_url
    self.driver = self._setup_driver()

  def _setup_driver(self) -> webdriver.Chrome:
    """Configura o Chrome em modo Headless para o ambiente do Colab."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        ' AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0'
        ' Safari/537.36'
    )
    return webdriver.Chrome(options=options)

  def fetch_html(self, path: str = '') -> str:
    """Acessa uma URL do site e retorna o HTML completo após a renderização."""
    url = f'{self.base_url}{path}'.strip('/')
    self.driver.get(url)
    time.sleep(3)  # Aguarda o carregamento dos scripts dinâmicos (JS)
    return self.driver.page_source

  def close(self):
    """Encerra a sessão do navegador."""
    self.driver.quit()

# Teste de Conectividade

Executa o teste de requisição

In [3]:
# Instancia o scraper e realiza o teste
scraper = NarutoArenaScraper()
try:
  html_content = scraper.fetch_html('')
  print(f'Sucesso! HTML capturado com {len(html_content)} caracteres.')
finally:
  scraper.close()

Sucesso! HTML capturado com 137018 caracteres.


# Captura Segura de Credenciais (Interativa)
Rode para preencher suas credenciais de forma segura em memória (sem salvar no código do notebook)

In [ ]:
import getpass
import os

# Solicita o login sem salvar em texto no código
NA_USER = input("Digite o E-mail/Usuário do Naruto-Arena: ")
NA_PASS = getpass.getpass("Digite a Senha (o texto ficará oculto): ")

print("Credenciais armazenadas com segurança em memória para a sessão atual!")

# Classe NarutoArenaScraper (Com Método de Login)

Autenticação adicionada.

In [8]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By


class NarutoArenaScraper:
  """Classe responsável pela navegação automatizada, autenticação e extração no Naruto-Arena utilizando Selenium Headless."""

  def __init__(self, base_url: str = 'https://www.naruto-arena.site/'):
    self.base_url = base_url.rstrip('/')
    self.driver = self._setup_driver()

  def _setup_driver(self) -> webdriver.Chrome:
    """Configura o Chrome em modo Headless otimizado para o ambiente do Colab."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        ' AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0'
        ' Safari/537.36'
    )
    return webdriver.Chrome(options=options)

  def login(self, username: str, password: str) -> bool:
    """Acessa a página /login, preenche o formulário via IDs e autentica a sessão."""
    try:
      login_url = f'{self.base_url}/login'
      self.driver.get(login_url)
      time.sleep(2)

      # Seletores reais identificados no HTML do site
      user_field = self.driver.find_element(By.ID, 'login-username')
      pass_field = self.driver.find_element(By.ID, 'login-password')

      user_field.clear()
      user_field.send_keys(username)
      pass_field.clear()
      pass_field.send_keys(password)

      # Submete o formulário enviando o comando 'ENTER' no campo de senha
      from selenium.webdriver.common.keys import Keys

      pass_field.send_keys(Keys.RETURN)

      time.sleep(4)
      print('Autenticação processada! Sessão ativa.')
      return True

    except Exception as e:
      print(f'Falha ao tentar realizar o login: {e}')
      return False

  def fetch_authenticated_page(self, path: str) -> str:
    """Navega para qualquer página interna do site reutilizando a sessão logada."""
    target_url = f'{self.base_url}/{path.lstrip("/")}'
    self.driver.get(target_url)
    time.sleep(3)  # Aguarda a renderização completa do JavaScript
    return self.driver.page_source

  def close(self):
    """Encerra a sessão do navegador."""
    self.driver.quit()

# Testando o Login com Credenciais Seguras
Execução no Colab para validar se o login funciona com as variáveis NA_USER e NA_PASS salvas em memória:

In [9]:
# Instancia o scraper e executa o teste de login
scraper = NarutoArenaScraper()

try:
  if scraper.login(NA_USER, NA_PASS):
    # Teste: Acessa a página principal após o login para verificar o HTML autenticado
    authenticated_html = scraper.fetch_authenticated_page('')
    print(
      f'Sucesso! Página capturada com {len(authenticated_html)} caracteres.'
    )
finally:
  scraper.close()

Autenticação processada! Sessão ativa.
Sucesso! Página capturada com 136092 caracteres.
